# Agentic RAG: Router–Retriever System with PDF and Web Search Tools

**Course-End Project** · Author: Alron Harry

This notebook builds a multi-agent **Agentic RAG** system with **CrewAI**:

| Agent | Role | Tools |
|---|---|---|
| **Router Agent** | Reads the question and decides the retrieval path: `pdf`, `web`, or `llm` (direct generation) | none (pure reasoning) |
| **Retriever Agent** | Runs the chosen tool, then writes a grounded, cited answer | `PDFSearchTool` (crewai_tools) **or** `TavilySearchResults` (LangChain) — only the one the router picked |

Knowledge base: *Attention Is All You Need* (Vaswani et al., 2017) — `trasformer_research_paper-dataset.pdf`.

**Notebook outline**
1. Setup & API keys
2. Trace logging (reasoning traceability)
3. Tools — PDF search & web search
4. Agents — roles, goals, backstories
5. Tasks — routing task & retrieval task
6. Orchestrator — coordination flow (router → retriever, with fallback)
7. Demo — PDF, web, and direct-LLM questions
8. Trace logs & visualisations
9. Why CrewAI? (framework justification)

## 1. Setup

In [ ]:
# Install dependencies (run once). Restart the kernel afterwards if prompted.
%pip install -q "crewai>=0.100" "crewai-tools>=0.33" langchain-community tavily-python pandas matplotlib

In [ ]:
import os, getpass

# Keys are read from the environment; if missing, you'll be prompted (input is hidden).
for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(f"Enter {key}: ")

# Model used by both agents (small, cheap, good at following structured-output instructions)
MODEL_NAME = os.environ.get("RAG_MODEL", "gpt-4o-mini")

# Path to the knowledge-base PDF (keep it in the same folder as this notebook)
PDF_PATH = "trasformer_research_paper-dataset.pdf"
assert os.path.exists(PDF_PATH), f"PDF not found at {PDF_PATH} — place it next to the notebook."

os.makedirs("logs", exist_ok=True)
print("Model:", MODEL_NAME, "| PDF:", PDF_PATH)

In [ ]:
import json, time, uuid, logging, textwrap, warnings
from datetime import datetime
from typing import Literal, Optional, Type, Any, List, Dict

from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from crewai_tools import PDFSearchTool
from langchain_community.tools.tavily_search import TavilySearchResults

warnings.filterwarnings("ignore", category=DeprecationWarning)
llm = LLM(model=MODEL_NAME, temperature=0)

## 2. Trace logging

Every interaction is recorded as a structured event so each agent's contribution can be audited later:

* `run_start` / `run_end` — the user question and the final answer
* `agent_step` — each thought / action / tool call an agent makes (via CrewAI `step_callback`)
* `tool_call` / `tool_result` — exact query sent to a tool and what came back (logged inside the tool wrappers)
* `task_complete` — the output of each task (via CrewAI `task_callback`)
* `route_decision` / `fallback` — orchestration decisions

Events go to memory (for the visualisations), to `logs/trace.jsonl` (machine-readable), and to `logs/agentic_rag.log` (human-readable).

In [ ]:
class TraceLogger:
    # Collects structured events for every run and mirrors them to disk.

    def __init__(self, jsonl_path="logs/trace.jsonl", log_path="logs/agentic_rag.log"):
        self.events: List[Dict[str, Any]] = []
        self.jsonl_path = jsonl_path
        self.run_id: Optional[str] = None
        self.logger = logging.getLogger("agentic_rag")
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            fh = logging.FileHandler(log_path, encoding="utf-8")
            fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s"))
            self.logger.addHandler(fh)

    def new_run(self, question: str) -> str:
        self.run_id = uuid.uuid4().hex[:8]
        self.log("Orchestrator", "run_start", question)
        return self.run_id

    def log(self, agent: str, event: str, content: Any, **extra):
        text = content if isinstance(content, str) else json.dumps(content, default=str)
        rec = {"run_id": self.run_id, "ts": time.time(),
               "time": datetime.now().strftime("%H:%M:%S.%f")[:-3],
               "agent": agent, "event": event, "content": text, **extra}
        self.events.append(rec)
        with open(self.jsonl_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, default=str) + "\n")
        self.logger.info(f"[{self.run_id}] {agent:<16} {event:<15} {text[:300]!r}")

    def print_run(self, run_id: str, width: int = 110):
        # Pretty-print the reasoning trace of one run.
        print(f"===== TRACE for run {run_id} =====")
        for e in [e for e in self.events if e["run_id"] == run_id]:
            body = textwrap.shorten(e["content"].replace("\n", " "), width=width, placeholder=" …")
            print(f"{e['time']}  {e['agent']:<16} {e['event']:<15} {body}")


tracer = TraceLogger()


def make_step_callback(agent_name: str):
    # CrewAI calls this after every agent step (AgentAction, ToolResult or AgentFinish).
    def _cb(step):
        kind = type(step).__name__
        payload = {}
        for attr in ("thought", "tool", "tool_input", "result", "output"):
            val = getattr(step, attr, None)
            if val:
                payload[attr] = str(val)[:1500]
        if not payload:
            payload["text"] = str(getattr(step, "text", step))[:1500]
        tracer.log(agent_name, "agent_step", payload, step_type=kind)
    return _cb


def task_callback(task_output):
    # CrewAI calls this when a task finishes.
    role = str(getattr(task_output, "agent", "unknown"))
    agent = {"Query Router": "Router Agent", "Retrieval Specialist": "Retriever Agent"}.get(role.strip(), role)
    tracer.log(agent, "task_complete", str(getattr(task_output, "raw", task_output))[:2000])

## 3. Tools

* **PDF search** — `crewai_tools.PDFSearchTool` chunks and embeds the paper into a vector store (OpenAI embeddings by default) and returns the most relevant passages for a query.
* **Web search** — LangChain's `TavilySearchResults` returns fresh results (title, URL, snippet) from the internet.

Both are wrapped in thin CrewAI `BaseTool` classes so that (a) every call and result is written to the trace log and (b) the web results are formatted with their URLs so the agent can cite them.

In [ ]:
# --- Underlying tools -------------------------------------------------------
pdf_search = PDFSearchTool(pdf=PDF_PATH)            # builds the vector index on first use
tavily = TavilySearchResults(max_results=4)          # LangChain Tavily wrapper


class QueryInput(BaseModel):
    query: str = Field(..., description="A focused search query.")


class PaperSearchTool(BaseTool):
    name: str = "transformer_paper_search"
    description: str = (
        "Semantic search over the research paper 'Attention Is All You Need' (the Transformer paper). "
        "Returns the most relevant text passages. Use for questions about the Transformer architecture, "
        "attention, training setup, hyperparameters, and reported results."
    )
    args_schema: Type[BaseModel] = QueryInput

    def _run(self, query: str) -> str:
        tracer.log("PDFSearchTool", "tool_call", {"query": query})
        result = pdf_search.run(query=query)
        tracer.log("PDFSearchTool", "tool_result", str(result)[:3000])
        return str(result)


class WebSearchTool(BaseTool):
    name: str = "web_search"
    description: str = (
        "Searches the live web with Tavily. Returns titles, URLs and snippets. Use for recent events, "
        "current facts, or anything not covered by the Transformer paper."
    )
    args_schema: Type[BaseModel] = QueryInput

    def _run(self, query: str) -> str:
        tracer.log("TavilySearch", "tool_call", {"query": query})
        try:
            results = tavily.invoke({"query": query})
        except Exception as exc:                       # network / quota errors shouldn't crash the run
            tracer.log("TavilySearch", "tool_error", str(exc))
            return f"Web search failed: {exc}"
        if isinstance(results, str):
            formatted = results
        else:
            formatted = "\n\n".join(
                f"[{i}] {r.get('title', '')}\nURL: {r.get('url', '')}\n{r.get('content', '')}"
                for i, r in enumerate(results, 1)
            )
        tracer.log("TavilySearch", "tool_result", formatted[:3000])
        return formatted


paper_tool = PaperSearchTool()
web_tool = WebSearchTool()
TOOLS = {"pdf": paper_tool, "web": web_tool}

In [ ]:
# Quick smoke test of both tools (also warms up the PDF vector index)
tracer.run_id = "smoke"
print(paper_tool._run("number of attention heads")[:600], "\n---")
print(web_tool._run("Transformer architecture latest developments")[:600])

## 4. Agents

**Router Agent** — a classifier. It never retrieves anything; it only reasons about *where* the answer lives and returns a structured decision (`RouteDecision`).

| Route | When |
|---|---|
| `pdf` | The question is about the Transformer paper: architecture, attention, positional encoding, training, BLEU results, authors… |
| `web` | Needs current / post-2017 / external information: news, latest models, prices, people today… |
| `llm` | General knowledge or conversational requests that need no retrieval (definitions, maths, rewording). |

**Retriever Agent** — receives the routed question, uses **only** the tool the router selected, and writes an answer grounded in the retrieved text with citations (page passages or URLs). If the evidence is insufficient it says so explicitly instead of guessing.

In [ ]:
class RouteDecision(BaseModel):
    route: Literal["pdf", "web", "llm"] = Field(..., description="Chosen retrieval path")
    reasoning: str = Field(..., description="One or two sentences explaining the choice")
    search_query: str = Field(..., description="Optimised query to send to the chosen tool")
    confidence: float = Field(..., ge=0, le=1, description="Confidence in the route, 0-1")


router_agent = Agent(
    role="Query Router",
    goal=("Classify each user question and choose the single best source to answer it: the Transformer "
          "research paper (pdf), the live web (web), or the LLM's own knowledge (llm)."),
    backstory=(
        "You are the dispatcher of a research assistant. The local knowledge base contains exactly one "
        "document: the 2017 paper 'Attention Is All You Need' that introduced the Transformer. You know "
        "its scope well: architecture (encoder/decoder stacks, multi-head and scaled dot-product attention, "
        "positional encodings, feed-forward layers), training details (WMT 2014 data, Adam, warmup, dropout, "
        "label smoothing), results (BLEU scores, English constituency parsing) and its authors. Anything "
        "recent, time-sensitive, or outside that paper must go to the web. Simple general-knowledge or "
        "conversational requests can be answered directly by the LLM. You never answer the question yourself."
    ),
    llm=llm,
    tools=[],
    allow_delegation=False,
    verbose=True,
    step_callback=make_step_callback("Router Agent"),
)


def make_retriever_agent(route: str) -> Agent:
    # Retriever gets ONLY the tool chosen by the router (none for the direct-LLM path).
    tools = [TOOLS[route]] if route in TOOLS else []
    source_hint = {
        "pdf": "the Transformer paper via transformer_paper_search",
        "web": "the live web via web_search",
        "llm": "your own general knowledge (no tools available)",
    }[route]
    return Agent(
        role="Retrieval Specialist",
        goal=f"Retrieve evidence from {source_hint} and write an accurate, well-grounded answer.",
        backstory=(
            "You are a meticulous research assistant. You always look at the evidence before you answer, "
            "quote or paraphrase only what the evidence supports, and cite your sources. If the evidence "
            "does not contain the answer you say 'INSUFFICIENT_EVIDENCE' rather than inventing facts."
        ),
        llm=llm,
        tools=tools,
        allow_delegation=False,
        max_iter=5,
        verbose=True,
        step_callback=make_step_callback("Retriever Agent"),
    )

## 5. Tasks

* **Routing task** → produces a `RouteDecision` (enforced with `output_pydantic`).
* **Retrieval task** → takes the original question **and** the router's optimised query, calls the tool, and returns the answer with a *Sources* section.

In [ ]:
def make_routing_task(question: str) -> Task:
    return Task(
        description=(
            f"User question: \"{question}\"\n\n"
            "Decide which source should answer it:\n"
            "- 'pdf'  -> the question concerns the Transformer paper (Attention Is All You Need).\n"
            "- 'web'  -> needs current, recent, or external information not in that paper.\n"
            "- 'llm'  -> simple general knowledge / conversational; no retrieval needed.\n"
            "Also rewrite the question into a concise search_query for the chosen tool, "
            "and give your confidence (0-1)."
        ),
        expected_output="A JSON object with fields: route, reasoning, search_query, confidence.",
        agent=router_agent,
        output_pydantic=RouteDecision,
    )


def make_retrieval_task(question: str, decision: RouteDecision, agent: Agent) -> Task:
    if decision.route == "llm":
        instructions = "Answer directly from your own knowledge. Be concise and accurate."
        sources = "Write 'Sources: LLM general knowledge (no retrieval)'."
    else:
        tool_name = TOOLS[decision.route].name
        instructions = (
            f"Use the `{tool_name}` tool with the query \"{decision.search_query}\" "
            "(you may refine and search again at most twice). Base your answer ONLY on the retrieved text."
        )
        sources = ("End with a 'Sources:' section listing the paper sections/passages used."
                   if decision.route == "pdf" else
                   "End with a 'Sources:' section listing the URLs used.")
    return Task(
        description=(
            f"Original user question: \"{question}\"\n"
            f"Router decision: route={decision.route}; reason={decision.reasoning}\n\n"
            f"{instructions}\n"
            "If the evidence does not answer the question, reply with exactly 'INSUFFICIENT_EVIDENCE' "
            "followed by one sentence explaining what is missing.\n"
            f"{sources}"
        ),
        expected_output="A clear 2-6 sentence answer grounded in the evidence, followed by a Sources section.",
        agent=agent,
    )

## 6. Orchestrator — coordination flow

```
            ┌──────────────┐  RouteDecision   ┌──────────────────┐   tool call   ┌──────────────────┐
 question ─▶│ Router Agent │ ───────────────▶ │ Orchestrator     │ ────────────▶ │ Retriever Agent  │──▶ grounded answer
            └──────────────┘ (route, query)   │ (picks the tool) │               │ PDF │ Web │ LLM   │
                                              └──────────────────┘               └──────────────────┘
                                                        ▲            INSUFFICIENT_EVIDENCE on pdf │
                                                        └──────────── fallback to web ◀──────────┘
```

1. The **Router crew** runs and returns a validated `RouteDecision`. If structured parsing fails, a keyword heuristic is used as a safety net (logged).
2. The orchestrator builds a **Retriever Agent equipped with only the chosen tool** — the router's decision is enforced, not just suggested.
3. The **Retriever crew** retrieves and answers.
4. **Fallback:** if the PDF path returns `INSUFFICIENT_EVIDENCE`, the orchestrator retries once on the web path (and logs it).

In [ ]:
PDF_KEYWORDS = ["transformer", "attention", "encoder", "decoder", "positional", "bleu", "wmt",
                "multi-head", "self-attention", "vaswani", "label smoothing", "warmup", "this paper"]
WEB_KEYWORDS = ["latest", "today", "current", "news", "2024", "2025", "2026", "recent", "price", "who is"]


def heuristic_route(question: str) -> RouteDecision:
    q = question.lower()
    if any(k in q for k in WEB_KEYWORDS):
        route = "web"
    elif any(k in q for k in PDF_KEYWORDS):
        route = "pdf"
    else:
        route = "llm"
    return RouteDecision(route=route, reasoning="Keyword heuristic fallback.", search_query=question, confidence=0.4)


class AgenticRAG:
    def __init__(self, enable_fallback: bool = True):
        self.enable_fallback = enable_fallback
        self.history: List[Dict[str, Any]] = []

    # ---- Stage 1: routing ----------------------------------------------------
    def route(self, question: str) -> RouteDecision:
        crew = Crew(agents=[router_agent], tasks=[make_routing_task(question)],
                    process=Process.sequential, verbose=True,
                    task_callback=task_callback, output_log_file="logs/crew_router.log")
        try:
            out = crew.kickoff()
            decision = out.pydantic if out.pydantic else RouteDecision.model_validate_json(out.raw)
        except Exception as exc:
            tracer.log("Orchestrator", "router_error", str(exc))
            decision = heuristic_route(question)
        tracer.log("Router Agent", "route_decision", decision.model_dump())
        return decision

    # ---- Stage 2: retrieval + answer -----------------------------------------
    def retrieve(self, question: str, decision: RouteDecision) -> str:
        agent = make_retriever_agent(decision.route)
        task = make_retrieval_task(question, decision, agent)
        crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=True,
                    task_callback=task_callback, output_log_file="logs/crew_retriever.log")
        return crew.kickoff().raw

    # ---- Full pipeline --------------------------------------------------------
    def ask(self, question: str) -> Dict[str, Any]:
        run_id = tracer.new_run(question)
        t0 = time.time()
        decision = self.route(question)
        answer = self.retrieve(question, decision)
        final_route = decision.route

        if self.enable_fallback and decision.route == "pdf" and "INSUFFICIENT_EVIDENCE" in answer:
            tracer.log("Orchestrator", "fallback", "PDF evidence insufficient -> retrying with web search")
            web_decision = RouteDecision(route="web", reasoning="Fallback: not found in the paper.",
                                         search_query=decision.search_query, confidence=decision.confidence)
            answer = self.retrieve(question, web_decision)
            final_route = "pdf→web"

        elapsed = round(time.time() - t0, 1)
        tracer.log("Orchestrator", "run_end", answer, route=final_route, seconds=elapsed)
        record = {"run_id": run_id, "question": question, "route": final_route,
                  "router_reasoning": decision.reasoning, "confidence": decision.confidence,
                  "answer": answer, "seconds": elapsed}
        self.history.append(record)
        return record


rag = AgenticRAG()


def show(record: Dict[str, Any]):
    print("\n" + "=" * 100)
    print(f"QUESTION : {record['question']}")
    print(f"ROUTE    : {record['route']}  (confidence {record['confidence']:.2f})")
    print(f"WHY      : {record['router_reasoning']}")
    print("-" * 100)
    print(record["answer"])
    print("=" * 100)

## 7. Demo

### 7.1 PDF path — questions answered from the Transformer paper

In [ ]:
r1 = rag.ask("What BLEU score did the Transformer achieve on the WMT 2014 English-to-German translation task?")
show(r1)

In [ ]:
r2 = rag.ask("How does multi-head attention work, and how many heads did the base model use?")
show(r2)

### 7.2 Web path — questions that need fresh information

In [ ]:
r3 = rag.ask("What are the latest open-weight large language models released this year?")
show(r3)

In [ ]:
r4 = rag.ask("Where do the authors of 'Attention Is All You Need' work today?")
show(r4)

### 7.3 Direct-LLM path — no retrieval needed

In [ ]:
r5 = rag.ask("In one sentence, what does RAG stand for in AI?")
show(r5)

### 7.4 Try your own question

In [ ]:
my_question = "What learning-rate schedule was used to train the Transformer?"   # edit me
show(rag.ask(my_question))

## 8. Trace logs & reasoning-trace visualisations

### 8.1 Step-by-step trace for one run
Shows exactly what each agent thought, which tool it called with what query, and what it produced.

In [ ]:
tracer.print_run(r1["run_id"])
print()
tracer.print_run(r3["run_id"])

### 8.2 Run summary

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)

summary = pd.DataFrame(rag.history)[["run_id", "question", "route", "confidence", "seconds", "router_reasoning"]]
summary

### 8.3 Full event log (all runs)

In [ ]:
events = pd.DataFrame([e for e in tracer.events if e["run_id"] != "smoke"])
events[["run_id", "time", "agent", "event", "content"]].assign(
    content=lambda d: d["content"].str.slice(0, 140))

### 8.4 Visualisations
* **Swimlane timeline** — when each agent / tool was active during a run (the hand-off from Router → Retriever → tool is visible).
* **Route distribution** and **events per agent** across all demo runs.

In [ ]:
import matplotlib.pyplot as plt

LANES = ["Orchestrator", "Router Agent", "Retriever Agent", "PDFSearchTool", "TavilySearch"]
COLORS = {"run_start": "#6b7280", "agent_step": "#2563eb", "route_decision": "#d97706",
          "tool_call": "#059669", "tool_result": "#10b981", "task_complete": "#7c3aed",
          "fallback": "#dc2626", "run_end": "#111827"}


def plot_swimlane(run_id: str, ax=None):
    ev = [e for e in tracer.events if e["run_id"] == run_id]
    if not ev:
        return
    t0 = ev[0]["ts"]
    ax = ax or plt.gca()
    for e in ev:
        lane = e["agent"] if e["agent"] in LANES else "Orchestrator"
        y = LANES.index(lane)
        ax.scatter(e["ts"] - t0, y, s=90, color=COLORS.get(e["event"], "#9ca3af"), zorder=3)
    # connect consecutive events to show the hand-offs
    xs = [e["ts"] - t0 for e in ev]
    ys = [LANES.index(e["agent"]) if e["agent"] in LANES else 0 for e in ev]
    ax.plot(xs, ys, color="#d1d5db", lw=1, zorder=1)
    q = next(e["content"] for e in ev if e["event"] == "run_start")
    ax.set_yticks(range(len(LANES)), LANES)
    ax.set_xlabel("seconds since question received")
    ax.set_title(textwrap.shorten(f"[{run_id}] {q}", 90), fontsize=10)
    ax.grid(axis="x", alpha=.3)


runs = [r["run_id"] for r in rag.history]
fig, axes = plt.subplots(len(runs), 1, figsize=(12, 2.6 * len(runs)), sharex=False)
axes = [axes] if len(runs) == 1 else axes
for rid, ax in zip(runs, axes):
    plot_swimlane(rid, ax)
handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=k) for k, c in COLORS.items()]
fig.legend(handles=handles, loc="upper center", ncol=len(COLORS), fontsize=8, frameon=False)
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.savefig("logs/swimlanes.png", dpi=120)
plt.show()

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
summary["route"].value_counts().plot.bar(ax=a1, color="#2563eb")
a1.set_title("Routes chosen across demo questions"); a1.set_ylabel("questions"); a1.tick_params(axis="x", rotation=0)

events.groupby(["agent", "event"]).size().unstack(fill_value=0).plot.barh(
    ax=a2, stacked=True, color=[COLORS.get(c, "#9ca3af") for c in events.groupby(["agent", "event"]).size().unstack().columns])
a2.set_title("Logged events per agent / tool"); a2.set_xlabel("events"); a2.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig("logs/summary_charts.png", dpi=120); plt.show()

In [ ]:
# Persist the run summary next to the JSONL trace
summary.to_csv("logs/run_summary.csv", index=False)
print("Saved: logs/trace.jsonl, logs/agentic_rag.log, logs/crew_router.log, logs/crew_retriever.log,"
      " logs/run_summary.csv, logs/swimlanes.png, logs/summary_charts.png")

## 9. Why CrewAI?

* **Role-first design.** CrewAI models agents as `role` + `goal` + `backstory`, which maps one-to-one onto the Router/Retriever responsibilities required here and keeps prompts readable.
* **Native tool ecosystem.** `PDFSearchTool` ships in `crewai_tools` (chunking + embedding + vector search in one line), and any LangChain tool such as `TavilySearchResults` can be wrapped as a `BaseTool`.
* **Structured outputs.** `output_pydantic` validates the router's decision, so the orchestrator branches on a typed object instead of parsing free text.
* **Built-in observability.** `step_callback`, `task_callback`, `verbose` and `output_log_file` give the hooks needed for the reasoning trace without extra infrastructure.
* **Alternatives considered.** *LangGraph* offers finer-grained graph control but needs more boilerplate for a two-agent pipeline; *AutoGen* is conversation-centric, which makes enforcing "use only the routed tool" harder. For a clear, auditable router → retriever pipeline, CrewAI is the simplest fit.

## 10. Conclusion

The system accepts a natural-language question, the **Router Agent** classifies it into `pdf` / `web` / `llm`, the orchestrator equips the **Retriever Agent** with exactly that tool, and the retriever returns a cited, source-grounded answer. Every step is logged to JSONL and visualised as swimlane timelines, giving full reasoning traceability. See `README.md` for architecture, responsibilities, and the challenges and trade-offs.